# 고향사랑기금, 잡아둔 돈의 17%만 썼다: 재현 노트북

데이터팝콘 테크블로그 글 「고향사랑기금은 정말 안 쓰고 있을까?」의 숫자와 차트 6개를 처음부터 다시 만듭니다.

- 원자료: 행정안전부 지방재정365 OpenAPI `세부사업별 세출현황(QWGJK)` 전국 243개 지자체 전수 스캔 결과 중 회계명·사업명에 '고향사랑'·'답례품'이 들어간 행 (2026-10-28 조회)
- 키 없이 실행됩니다. 수집해 둔 파일을 공개 저장소에서 읽습니다. 직접 다시 수집하려면 맨 아래 부록을 보세요.
- `런타임 → 모두 실행`

In [ ]:
# 한글 폰트 (Colab)
import os, sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.font_manager as fm
for f in fm.findSystemFonts():
    if 'Nanum' in f: fm.fontManager.addfont(f)
names = {f.name for f in fm.fontManager.ttflist}
mpl.rcParams['font.family'] = 'NanumGothic' if 'NanumGothic' in names else 'sans-serif'
mpl.rcParams['axes.unicode_minus'] = False
ORANGE, BLUE, GREY, LIGHT = '#E8590C', '#1C7ED6', '#868E96', '#DEE2E6'
SRC = '자료: 행정안전부 지방재정365 세부사업별 세출현황 · 2026-10-28 조회 · 데이터팝콘'

In [ ]:
import json, re, io, urllib.request
import pandas as pd, numpy as np
BASE = os.environ.get('DATA_DIR') or 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/hometown-love-fund-spending/data'
def read_text(name):
    if BASE.startswith('http'):
        return urllib.request.urlopen(f'{BASE}/{name}').read().decode('utf-8')
    return open(os.path.join(BASE, name), encoding='utf-8').read()
SNAPS = ['2023_20231231', '2024_20241027', '2024_20241231', '2025_20251027', '2025_20251231', '2026_20261027']
rows = []
for s in SNAPS:
    for line in read_text(f'QWGJK_gohyang_{s}.jsonl').splitlines():
        if line.strip():
            r = json.loads(line); r['snap'] = s.split('_')[1]; rows.append(r)
df = pd.DataFrame(rows)
for c in ['bdg_cash_amt', 'ep_amt']: df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
df = df.drop_duplicates(['snap', 'laf_cd', 'acnt_dv_cd', 'acnt_dv_nm', 'dept_cd', 'dbiz_cd'])
print(len(df), '행'); df.groupby('snap').size()

## Step 2. 고향사랑기금 회계만 골라내고, 사업을 네 종류로 나눕니다

- 회계명에 '고향사랑'이 들어가면 고향사랑기금 회계입니다(지자체마다 '고향사랑기금', '안성시 고향사랑기금' 등으로 이름이 다릅니다).
- **예치(적립)**: 기금예치·여유자금 예탁·내부거래·보전지출. 돈이 기금 밖 금고나 다른 회계로 옮겨갈 뿐이라 실제 집행에서 뺍니다.
- 나머지는 사업명으로 '운영·답례품', '포괄 기금사업'(이름만으로 내용을 모름), '주민사업'으로 나눕니다. 이름 기준 추정입니다.

In [ ]:
df['is_fund'] = df.acnt_dv_nm.str.contains('고향사랑')
def kind(n):
    core = re.sub(r'\(.*?\)', '', n).strip()
    if re.search(r'예치|예탁|적립|예수|여유자금|내부거래|보전지출|전출금', n): return '예치(적립)'
    if re.search(r'답례', core): return '운영·답례품'
    if re.search(r'기금\s*(사업|운영|운용|지원|관리|모집)|^고향사랑\s*기금$|^고향사랑\s*활성화$', core) and not re.search(r'지원사업|센터|버스|의료|돌봄|장학', core): return '포괄 기금사업'
    if re.search(r'기부제|기부금|홍보|모금|기부자', core): return '운영·답례품'
    return '주민사업'
df['kind'] = df.dbiz_nm.fillna('').map(kind)
f = df[df.is_fund]
g = f.pivot_table(index=['snap', 'laf_cd'], columns='kind', values=['bdg_cash_amt', 'ep_amt'], aggfunc='sum', fill_value=0)
g.columns = [f'{a}_{b}' for a, b in g.columns]; g = g.reset_index()
for k in ['예치(적립)', '운영·답례품', '포괄 기금사업', '주민사업']:
    for a in ['bdg_cash_amt', 'ep_amt']:
        if f'{a}_{k}' not in g: g[f'{a}_{k}'] = 0
g['fund_bdg'] = g.filter(like='bdg_cash_amt_').sum(axis=1)
g['real_ep'] = g.filter(like='ep_amt_').sum(axis=1) - g['ep_amt_예치(적립)']
g['reserve_bdg'] = g['bdg_cash_amt_예치(적립)']

## Step 3. 지자체 243곳을 시점마다 맞춥니다

2026년에는 행정구역이 바뀌었습니다(광주·전남 통합 → 전남광주본청, 인천 중구·동구 → 영종·제물포·검단구). 2023년 군위군은 경북 소속입니다. 기금 회계가 아예 없는 지자체도 '0원'으로 세려면 지자체 목록을 기준으로 붙여야 합니다.

In [ ]:
laf = pd.read_csv(io.StringIO(read_text('laf_list.csv')), dtype=str); laf.columns = ['wa', 'laf_cd', 'name']
extra = pd.DataFrame([['경북','4771000','경북군위군'],['인천','2811500','인천영종구'],['인천','2812500','인천제물포구'],['인천','2818500','인천검단구'],['전남광주','6100000','전남광주본청']], columns=laf.columns)
dgw = laf[laf.name.eq('대구군위군')].laf_cd.tolist()
def universe(s):
    u = set(laf.laf_cd)
    if s.startswith('2023'): u = (u - set(dgw)) | {'4771000'}
    if s.startswith('2026'): u = (u - {'2811000','2812000','2900000','4600000'}) | {'2811500','2812500','2818500','6100000'}
    return sorted(u)
lafx = pd.concat([laf, extra], ignore_index=True)
snaps = sorted(df.snap.unique())
base = pd.DataFrame([(s, c) for s in snaps for c in universe(s)], columns=['snap', 'laf_cd'])
p = base.merge(g, how='left').fillna(0).merge(lafx, on='laf_cd', how='left')
p['has_fund'] = p.fund_bdg > 0
p['rate'] = np.where(p.has_fund, p.real_ep / p.fund_bdg, np.nan)
p.groupby('snap').size()

## Step 4. 전국 합계: 예산 중 얼마가 실제로 나갔나

In [ ]:
E = 1e8
tot = p.groupby('snap').agg(fund_bdg=('fund_bdg','sum'), reserve=('reserve_bdg','sum'), real_ep=('real_ep','sum'),
                             n_fund=('has_fund','sum'), n_zero=('real_ep', lambda s: 0)).reset_index()
tot['n_zero'] = [int((p[(p.snap==s)&p.has_fund].real_ep <= 0).sum()) for s in tot.snap]
tot['지출률'] = tot.real_ep / tot.fund_bdg
tot['비예치 대비'] = tot.real_ep / (tot.fund_bdg - tot.reserve)
show = tot.assign(**{c: (tot[c]/E).round(1) for c in ['fund_bdg','reserve','real_ep']})
show

In [ ]:
ye = tot[tot.snap.str.endswith('1231')].copy(); ye['yr'] = ye.snap.str[:4]
fig, ax = plt.subplots(figsize=(9, 5.2)); x = np.arange(len(ye)); w = .38
ax.bar(x-w/2, ye.fund_bdg/E, w, color=LIGHT, label='기금 예산현액')
ax.bar(x-w/2, ye.reserve/E, w, color=GREY, label='└ 그중 예치·예탁·내부거래')
ax.bar(x+w/2, ye.real_ep/E, w, color=ORANGE, label='실제 지출')
for i, r in enumerate(ye.itertuples()):
    ax.text(i-w/2, r.fund_bdg/E+30, f'{r.fund_bdg/E:,.0f}억', ha='center')
    ax.text(i+w/2, r.real_ep/E+30, f'{r.real_ep/E:,.0f}억\n({r.real_ep/r.fund_bdg:.0%})', ha='center', color=ORANGE, fontweight='bold')
ax.set_xticks(x, ye.yr+'년 말'); ax.set_yticks([]); [ax.spines[s].set_visible(False) for s in ['top','right','left']]
ax.legend(frameon=False, loc='upper left'); ax.set_title('2025년 말 기금 예산 2,549억 중 실제로 나간 돈은 437억(17%)', loc='left', fontsize=13, fontweight='bold')
fig.text(0.01, -0.02, SRC, fontsize=8, color=GREY); plt.savefig('chart1.png', dpi=200, bbox_inches='tight'); plt.show()

## Step 5. 같은 날짜(10월 27일)끼리 비교합니다

In [ ]:
yd = tot[tot.snap.str.endswith('1027')].copy(); yd['yr'] = yd.snap.str[:4]
fig, ax = plt.subplots(figsize=(9, 5)); x = np.arange(len(yd))
ax.bar(x-w/2, yd.fund_bdg/E, w, color=LIGHT, label='기금 예산현액')
ax.bar(x+w/2, yd.real_ep/E, w, color=ORANGE, label='실제 지출 (1/1~10/27 누계)')
for i, r in enumerate(yd.itertuples()):
    ax.text(i-w/2, r.fund_bdg/E+30, f'{r.fund_bdg/E:,.0f}억', ha='center')
    ax.text(i+w/2, r.real_ep/E+30, f'{r.real_ep/E:,.0f}억\n({r.real_ep/r.fund_bdg:.0%})', ha='center', color=ORANGE, fontweight='bold')
ax.set_xticks(x, yd.yr+'년 10/27 기준'); ax.set_yticks([]); [ax.spines[s].set_visible(False) for s in ['top','right','left']]
ax.legend(frameon=False, loc='upper left'); ax.set_title('같은 날짜로 맞추면 지출률은 6% → 11% → 16%', loc='left', fontsize=13, fontweight='bold')
fig.text(0.01, -0.02, SRC, fontsize=8, color=GREY); plt.savefig('chart2.png', dpi=200, bbox_inches='tight'); plt.show()
print(yd[['yr','n_zero']])

## Step 6. 지자체별로 봅니다: 한 푼도 안 쓴 곳은 어디인가

In [ ]:
y = p[(p.snap=='20251231') & p.has_fund].copy()
print('기금 지출 0원:', int((y.real_ep<=0).sum()), '/ 지출률 10% 미만:', int((y.rate<.10).sum()), '/ 중앙값:', round(y.rate.median()*100,1), '%')
bins = [-1, 1e-12, .1, .2, .3, .5, 1.01]; labels = ['0%', '0~10%', '10~20%', '20~30%', '30~50%', '50% 이상']
cnt = pd.cut(y.rate.clip(upper=1), bins=bins, labels=labels).value_counts().reindex(labels)
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.bar(labels, cnt.values, color=[ORANGE]+[LIGHT]*5)
for i, v in enumerate(cnt.values): ax.text(i, v+1, f'{v}곳', ha='center')
ax.set_yticks([]); [ax.spines[s].set_visible(False) for s in ['top','right','left']]
ax.set_title(f'243곳 중 {int((y.real_ep<=0).sum())}곳은 2025년에 기금에서 1원도 쓰지 않았다', loc='left', fontsize=13, fontweight='bold')
ax.set_xlabel('2025년 기금 지출률 (실제 지출 ÷ 기금 예산현액)')
fig.text(0.01, -0.04, SRC, fontsize=8, color=GREY); plt.savefig('chart3.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
r = y.assign(zero=y.real_ep<=0).groupby('wa').agg(n=('zero','size'), z=('zero','sum'), bdg=('fund_bdg','sum')).assign(zr=lambda d: d.z/d.n).sort_values(['zr','bdg'])
fig, ax = plt.subplots(figsize=(9, 6.2))
ax.barh(r.index, r.zr*100, color=[ORANGE if v>=.3 else (BLUE if v<=.1 else LIGHT) for v in r.zr])
for i, (k, v) in enumerate(r.iterrows()): ax.text(v.zr*100+1, i, f'{int(v.z)}/{int(v.n)}곳 (기금 예산 {v.bdg/E:,.0f}억)', va='center', fontsize=9)
ax.set_xticks([]); [ax.spines[s].set_visible(False) for s in ['top','right','bottom']]
ax.set_title('기금에서 1원도 안 쓴 지자체 비율: 울산·대구 절반, 전남·전북 1곳', loc='left', fontsize=13, fontweight='bold')
fig.text(0.01, -0.02, SRC+' · 2025년 말 기준', fontsize=8, color=GREY); plt.savefig('chart4.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
# 재정자립도와 관계
fin = pd.DataFrame(json.loads(read_text('lofin_JFIED_2024.json'))['rows'])[['laf_cd','rate2']]
yy = y.merge(fin, on='laf_cd').dropna(subset=['rate2'])
yy['tert'] = pd.qcut(yy.rate2.astype(float), 3, labels=['하위','중위','상위'])
print(yy.groupby('tert', observed=True).rate.median().round(3))
print('순위상관', round(yy[['rate2','rate']].astype(float).corr(method='spearman').iloc[0,1], 3))

## Step 7. 쓴 돈은 어디로 갔나 (사업명 기준 추정)

In [ ]:
r25 = f[f.snap=='20251231'].groupby('kind').ep_amt.sum()/E
order = ['주민사업', '포괄 기금사업', '운영·답례품', '예치(적립)']; cols = [BLUE, '#74C0FC', ORANGE, GREY]
fig, ax = plt.subplots(figsize=(10, 2.6)); left = 0
for k, c in zip(order, cols):
    v = r25.get(k, 0); ax.barh(0, v, left=left, color=c)
    ax.text(left+v/2, 0, f'{k}\n{v:,.0f}억', ha='center', va='center', color='white', fontweight='bold', fontsize=10); left += v
ax.axis('off'); ax.set_title(f'2025년 기금 회계 지출 {r25.sum():,.0f}억 중 이름에 내용이 드러나는 주민사업은 {r25["주민사업"]:,.0f}억', loc='left', fontsize=13, fontweight='bold')
fig.text(0.01, -0.08, SRC+' · 세부사업명 기준 분류(추정)', fontsize=8, color=GREY); plt.savefig('chart5.png', dpi=200, bbox_inches='tight'); plt.show()

## Step 8. 사람들은 무엇을 검색하나 (네이버 검색광고 월간 검색수)

In [ ]:
sv = pd.read_csv(io.StringIO(read_text('search_volume.csv')))
sv2 = sv[sv.keyword.str.contains('네이버페이|답례품 추천|세액공제|이벤트|사용처|집행')].copy()
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(sv2.keyword[::-1], sv2.monthly_searches[::-1], color=[GREY if '사용처' in k or '집행' in k else ORANGE for k in sv2.keyword[::-1]])
for i, v in enumerate(sv2.monthly_searches[::-1]): ax.text(v+30, i, '10 미만' if v<=10 else f'{v:,}', va='center')
ax.set_xticks([]); [ax.spines[s].set_visible(False) for s in ['top','right','bottom']]
ax.set_title('답례품·세액공제는 찾고, 기부금 쓰임새는 거의 안 찾는다', loc='left', fontsize=13, fontweight='bold')
fig.text(0.01, -0.03, '자료: 네이버 검색광고 키워드도구 월간 검색수 · 2026-10-28 조회 · 데이터팝콘', fontsize=8, color=GREY); plt.savefig('chart6.png', dpi=200, bbox_inches='tight'); plt.show()

## Step 9. 검증: 지자체가 직접 공개한 원문과 대조

In [ ]:
cc = pd.read_csv(io.StringIO(read_text('crosscheck_disclosure_vs_lofin.csv')))
cc['차이(억)'] = (cc['lofin_실집행억'] - cc['공시_기금사용억']).round(2)
cc

## 부록. 직접 다시 수집하기 (기본 꺼짐)

1. [지방재정365](https://www.lofin365.go.kr) 회원가입(휴대폰 본인인증) → 마이페이지 → 인증키발급
2. Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `LOFIN_API_KEY` 저장
3. 아래 `RUN_COLLECT = True`로 바꾸고 실행. 연도마다 약 48만 행(1,000행 × 480페이지)을 훑습니다.

In [ ]:
RUN_COLLECT = False
if RUN_COLLECT:
    import time
    try:
        from google.colab import userdata; KEY = userdata.get('LOFIN_API_KEY')
    except Exception: KEY = os.environ['LOFIN_API_KEY']
    def page(fy, ymd, i):
        u = f'https://www.lofin365.go.kr/lf/hub/QWGJK?Key={KEY}&Type=json&pIndex={i}&pSize=1000&fyr={fy}&exe_ymd={ymd}'
        return json.loads(urllib.request.urlopen(u).read())['QWGJK']
    fy, ymd = '2025', '20251231'
    head = page(fy, ymd, 1); total = head[0]['head'][0]['list_total_count']; keep = []
    for i in range(1, total//1000 + 2):
        for r in page(fy, ymd, i)[1]['row']:
            if '고향사랑' in (r.get('acnt_dv_nm') or '') or re.search('고향사랑|답례품', r.get('dbiz_nm') or ''): keep.append(r)
        time.sleep(0.1)
    print(total, '행 중', len(keep), '행 보관')